In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
train = pd.read_csv(r"/workspaces/alpha-centauri-rescue/data/raw/train (1).csv")
test = pd.read_csv(r"/workspaces/alpha-centauri-rescue/data/raw/test (1).csv")

In [17]:
train.shape

(8693, 14)

In [18]:
train.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [19]:
train.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='str')

In [20]:
test.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name'],
      dtype='str')

In [21]:
train["Transported"].value_counts()

Transported
True     4378
False    4315
Name: count, dtype: int64

Feature Engineering

In [22]:
for df in [train, test]:
    # 1. Split Cabin into Deck, Number, and Side (P = Port, S = Starboard)
    df[['CabinDeck', 'CabinNum', 'CabinSide']] = df['Cabin'].str.split('/', expand=True)
    df['CabinNum'] = df['CabinNum'].astype(float)
    
    # 2. Extract Group ID from PassengerId (gggg_pp -> gggg)
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
    
    # 3. Calculate Total Spending across all luxury amenities
    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpend'] = df[spending_cols].sum(axis=1)
    
    # 4. Create a binary indicator: Did they spend any money at all?
    df['IsSpender'] = (df['TotalSpend'] > 0).astype(int)

# Calculate Group Size by combining train and test groups
all_groups = pd.concat([train['Group'], test['Group']])
group_sizes = all_groups.value_counts().to_dict()

for df in [train, test]:
    df['GroupSize'] = df['Group'].map(group_sizes)


Handle the Missing Values

In [23]:
# Identify feature types
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNum', 'TotalSpend', 'GroupSize']
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'CabinDeck', 'CabinSide']

# Fill missing values
for col in num_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

for col in cat_cols:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    test[col] = test[col].fillna(mode_val)


Dropping Unnecessary Columns & Converting Categoricals

In [24]:
# Separate features and target from train
X = train.drop(columns=['PassengerId', 'Cabin', 'Name', 'Group', 'Transported'])
y = train['Transported'].astype(int)  # Convert True/False to 1/0

# Keep PassengerId for the final submission file, drop the rest from test
X_test = test.drop(columns=['PassengerId', 'Cabin', 'Name', 'Group'])

# Convert boolean/categorical columns into numerical indicators (0 and 1)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Align columns to ensure both datasets match exactly
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)


Model Training

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Split our training data to evaluate locally
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Check your baseline accuracy
val_preds = model.predict(X_val)
print(f"Baseline Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")


Baseline Validation Accuracy: 0.7849


Implementing lightGBM

In [26]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import lightgbm as lgb

# Clean up column names (LightGBM can complain about special characters like spaces/brackets)
X.columns = [str(col).replace('[', '').replace(']', '').replace('<', '') for col in X.columns]
X_test.columns = [str(col).replace('[', '').replace(']', '').replace('<', '') for col in X_test.columns]

# Set up 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

# LightGBM Parameters optimized for tabular data
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_error',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'random_state': 42,
    'verbose': -1
}

print("Starting LightGBM Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Train the model
    model = lgb.LGBMClassifier(**lgb_params, n_estimators=500)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Predict on validation split
    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    
    # Score the fold
    score = accuracy_score(y_val, val_preds)
    fold_scores.append(score)
    print(f"Fold {fold + 1} Accuracy: {score:.4f}")
    
    # Accumulate test predictions (we average these across folds later)
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

print(f"\nMean CV Accuracy: {np.mean(fold_scores):.4f}")


Starting LightGBM Cross-Validation...


/usr/local/python/3.14.2/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 Accuracy: 0.8160


/usr/local/python/3.14.2/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 Accuracy: 0.8114


...

Fold 3 Accuracy: 0.8229


/usr/local/python/3.14.2/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4 Accuracy: 0.8216


/usr/local/python/3.14.2/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 Accuracy: 0.8044

Mean CV Accuracy: 0.8153


In [28]:


# Convert probabilities to binary predictions (True/False)
final_preds = (test_preds >= 0.5).astype(bool)

# Build the submission dataframe 
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': final_preds
})

# Save to your output directory
submission.to_csv('/workspaces/alpha-centauri-rescue/data/raw/sample_submission.csv', index=False)
